In [0]:
import yaml
local_config_path = "/Workspace/Repos/adb-tetiana@startsteps.org/E-commerce-Orders-Delta-Pipeline/config/projects_config.yml"

with open(local_config_path, "r") as f:
    config = yaml.safe_load(f)

catalog_name = config["catalog"]
schema_name = config["schema"]   
landing_volume = config["landing_volume"]
checkpoint_volume = config["checkpoint_volume"]

landing_path = config["paths"]["landing_orders"]
checkpoint_path = config["paths"]["autoloader_checkpoint"]

bronze_table = config["tables"]["bronze_orders"]
silver_table = config["tables"]["siver_orders"]
gold_daily_orders_summary = config["tables"]["gold_daily_orders_summary"]
gold_status_summary = config["tables"]["gold_status_summary"]
gold_shipping_method_summary = config["tables"]["gold_shipping_method_summary"]

high_value_threshold = config["rules"]["high_value_threshold"]

print(f"landing_path: {landing_path}")
print(f"checkpoint_path: {checkpoint_path}")
print(f"bronze_table: {bronze_table}")
print(f"bronze_table: {bronze_table}")
print(f"silver_table: {silver_table}")
print(f"gold_daily_orders_summary: {gold_daily_orders_summary}")
print(f"gold_status_summary: {gold_status_summary}")
print(f"gold_shipping_method_summary: {gold_shipping_method_summary}")
print(f"high_value_threshold: {high_value_threshold}")


In [0]:
bronze_df = spark.table(bronze_table)

display(bronze_df)


In [0]:

display(bronze_df.groupBy("order_status").count())
display(bronze_df.groupBy("shipping_method").count())
display(bronze_df.groupBy("payment_status").count())

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
silver_base_df = (
    bronze_df
    .select(
        "customer_id",
        "order_id",
        "order_status",
        "order_amount",
        "order_date",
        "shipping_method",
        "payment_status",
        "delivery_country",
        "last_updated_ts"
        )
    .withColumn("order_amount", F.col("order_amount").cast("double"))
    .withColumn("order_date", F.col("order_date").cast("date"))
    .withColumn("last_updated_ts", F.to_timestamp("last_updated_ts"))
)

In [0]:
silver_filtered_df = (
    silver_base_df
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("order_amount").isNotNull())
    .filter(F.col("order_date").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    .filter(F.col("order_amount") > 0)
)

In [0]:
silver_standart_df = (
    silver_filtered_df
    .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
    .withColumn("shipping_method", F.upper(F.trim(F.col("shipping_method"))))
    .withColumn("payment_status", F.upper(F.trim(F.col("payment_status"))))
    .withColumn("delivery_country", F.upper(F.trim(F.col("delivery_country"))))
)

In [0]:
#  keep the latest record for each order
window_spec = Window.partitionBy("order_id").orderBy(F.col("last_updated_ts").desc())

In [0]:
silver_dedup_df = (
    silver_standart_df
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

In [0]:
silver_df = (
    silver_dedup_df
    .withColumn("order_date", F.col("order_date"))
    .withColumn(
        "is_high_value_order",
        F.when(F.col("order_amount") > high_value_threshold, 1)
        .otherwise(0)
    )
)

In [0]:
silver_df.write.mode("overwrite").saveAsTable(silver_table)
display(spark.table(silver_table))